# 11-тарау. Эмбеддингтер және назар аудару

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 11.1. Токеннен векторға

In [ ]:
import math
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(0)
emb = nn.Embedding(num_embeddings=10, embedding_dim=4)   # 10 токен, 4 сан
ids = torch.tensor([3, 7, 3])
print(emb(ids))
print(emb.weight.shape)

In [ ]:
one_hot = F.one_hot(ids, num_classes=10).float()
print(torch.allclose(one_hot @ emb.weight, emb(ids)))

In [ ]:
words = {"патша":    torch.tensor([0.9,  0.8, 1.0]),
         "ханшайым": torch.tensor([0.9, -0.8, 1.0]),
         "ер":       torch.tensor([0.1,  0.8, 1.0]),
         "әйел":     torch.tensor([0.1, -0.8, 1.0]),
         "тақ":      torch.tensor([0.8,  0.0, 0.0])}
target = words["патша"] - words["ер"] + words["әйел"]
for w, v in words.items():
    sim = F.cosine_similarity(target, v, dim=0).item()
    print(f"{w:9s} {sim:.3f}")

## 11.2. Сөздің орны

In [ ]:
def sinusoidal_positions(n_pos, dim):
    pos = torch.arange(n_pos).unsqueeze(1)       # (n_pos, 1)
    i = torch.arange(0, dim, 2)                  # жұп индекстер
    angle = pos / (10000 ** (i / dim))
    pe = torch.zeros(n_pos, dim)
    pe[:, 0::2] = torch.sin(angle)
    pe[:, 1::2] = torch.cos(angle)
    return pe

pe = sinusoidal_positions(64, 32)
print(pe.shape)
print(pe[:3, :6].round(decimals=3))

## 11.3. Назар аудару механизмі

In [ ]:
tokens = ["Айгерім", "кітап", "оқыды", "ол"]
K = torch.tensor([[1., 0, 0, 0],     # Айгерім: адам
                  [0., 1, 0, 0],     # кітап: зат
                  [0., 0, 1, 0],     # оқыды: әрекет
                  [0., 0, 0, 1]])    # ол: есімдік
Q = torch.tensor([[0., 0, 2, 0],     # Айгерім: «не істеді?»
                  [0., 0, 2, 0],     # кітап: «онымен не болды?»
                  [2., 2, 0, 0],     # оқыды: «кім? нені?»
                  [4., 0, 0, 0]])    # ол: «кім туралы?»
V = torch.eye(4)

scores = Q @ K.T / math.sqrt(4)
weights = torch.softmax(scores, dim=-1)
for tok, row in zip(tokens, weights):
    print(f"{tok:8s}", [f"{w:.2f}" for w in row.tolist()])

In [ ]:
n = len(tokens)
mask = torch.tril(torch.ones(n, n))      # төменгі үшбұрыш
masked = scores.masked_fill(mask == 0, float("-inf"))
causal_weights = torch.softmax(masked, dim=-1)
print(causal_weights.round(decimals=2))

## 11.4. Көп басты назар

In [ ]:
def attention(q, k, v, causal=True):
    d = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)
    if causal:
        n = q.size(-2)
        mask = torch.tril(torch.ones(n, n, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float("-inf"))
    return torch.softmax(scores, dim=-1) @ v

torch.manual_seed(0)
B, T, C, H = 2, 5, 16, 4         # топтама, ұзындық, өлшем, бастар
x = torch.randn(B, T, C)
qkv = nn.Linear(C, 3 * C)        # үш проекция бір қабатта
q, k, v = qkv(x).split(C, dim=2)
# (B, T, C) → (B, H, T, C/H): әр бас өз бөлігін алады
q, k, v = [t.view(B, T, H, C // H).transpose(1, 2) for t in (q, k, v)]
out = attention(q, k, v)
print(out.shape)
ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print(torch.allclose(out, ref, atol=1e-6))
out = out.transpose(1, 2).reshape(B, T, C)   # бастарды біріктіру
print(out.shape)